# Auto Insurance Risk & Premium Prediction Analysis
This notebook shows a beginner-friendly project for auto insurance risk and premium prediction using Python, pandas, seaborn, and scikit-learn.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, mean_absolute_error, mean_squared_error, r2_score

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)


In [ ]:
# Load the dataset
file_path = '../data/insurance_data.csv'
df = pd.read_csv(file_path)
df.head()

In [ ]:
# Exploratory Data Analysis
print('Shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())
print('\nData types:')
print(df.dtypes)

print('\nSummary statistics:')
print(df.describe())

plt.figure()
sns.countplot(x='claim_risk', data=df)
plt.title('High Risk vs Low Risk Customers')
plt.show()

plt.figure()
sns.histplot(df['premium'], kde=True, color='green')
plt.title('Premium Distribution')
plt.xlabel('Premium')
plt.show()

plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation')
plt.show()

In [ ]:
# Data Cleaning and Preprocessing
# Remove duplicate rows and fill missing values if needed

print('Duplicates:', df.duplicated().sum())
df = df.drop_duplicates()

# For this dataset there are no missing values, but we keep a simple check
if df.isnull().sum().sum() > 0:
    df = df.dropna()

# Encode categorical values

df['sex'] = df['sex'].map({'female': 0, 'male': 1})
df['smoker'] = df['smoker'].map({'no': 0, 'yes': 1})
region_dummies = pd.get_dummies(df['region'], prefix='region', drop_first=True)
df = pd.concat([df.drop(columns=['region']), region_dummies], axis=1)

# Feature engineering

df['age_group'] = pd.cut(df['age'], bins=[17, 25, 35, 50, 70], labels=['18-25', '26-35', '36-50', '51-70'])
df['risk_score'] = df['accident_history'] + df['smoker'] + (df['vehicle_age'] > 8).astype(int)

print(df[['age', 'age_group', 'vehicle_age', 'risk_score']].head())

In [ ]:
# Build Classification Models
feature_cols = [
    'age', 'sex', 'bmi', 'children', 'smoker',
    'vehicle_age', 'annual_mileage', 'accident_history',
    'credit_score', 'driving_experience',
    'region_northwest', 'region_southeast', 'region_southwest',
    'risk_score'
]

X = df[feature_cols]
y = df['claim_risk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

log_model = LogisticRegression(max_iter=200, random_state=42)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

log_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

print('Classification models trained.')

In [ ]:
# Evaluate Classification Models
models = {'Logistic Regression': log_model, 'Random Forest': rf_model}

for name, model in models.items():
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    print(f'\n{name}')
    print('Accuracy:', accuracy_score(y_test, preds))
    print('Precision:', precision_score(y_test, preds, zero_division=0))
    print('Recall:', recall_score(y_test, preds, zero_division=0))
    print('F1-score:', f1_score(y_test, preds, zero_division=0))
    print('ROC-AUC:', roc_auc_score(y_test, probs))

In [ ]:
# Build Regression Models
# Use the same input features to predict premium

y_reg = df['premium']
X_train_rg, X_test_rg, y_train_rg, y_test_rg = train_test_split(X, y_reg, test_size=0.2, random_state=42)

lin_model = LinearRegression()
rf_reg_model = RandomForestRegressor(n_estimators=100, random_state=42)

lin_model.fit(X_train_rg, y_train_rg)
rf_reg_model.fit(X_train_rg, y_train_rg)

print('Regression models trained.')

In [ ]:
# Evaluate Regression Models
reg_models = {'Linear Regression': lin_model, 'Random Forest Regressor': rf_reg_model}

for name, model in reg_models.items():
    preds = model.predict(X_test_rg)
    print(f'\n{name}')
    print('MAE:', mean_absolute_error(y_test_rg, preds))
    print('RMSE:', mean_squared_error(y_test_rg, preds, squared=False))
    print('R2 score:', r2_score(y_test_rg, preds))

In [ ]:
# Visualizations
plt.figure()
sns.boxplot(x='smoker', y='premium', data=df)
plt.title('Premium by Smoker Status')
plt.show()

plt.figure()
sns.scatterplot(x='vehicle_age', y='premium', hue='claim_risk', data=df, palette='Set2')
plt.title('Premium vs Vehicle Age')
plt.show()

plt.figure(figsize=(8, 4))
sns.barplot(x='age_group', y='claim_risk', data=df, estimator=np.mean)
plt.title('Average Claim Risk by Age Group')
plt.ylabel('High Risk Rate')
plt.show()

## Business Insights

- Customers with high accident history and smoker status are more likely to be labeled high risk, which supports underwriting risk assessment.
- Premium estimates increase with vehicle age and lower credit score, indicating consumer segments that may need higher pricing.
- Customer segmentation by risk score can guide targeted policy offers and premium discounts for lower-risk groups.
- Underwriting teams can use the model results to balance claim risk and premium pricing while improving profitability.


In [4]:
from pathlib import Path

# Generate the cleaned insurance dataset if it does not exist
csv_path = Path('../data/insurance_data.csv')
source_path = Path('../../insurance.csv')

if not csv_path.exists():
    raw_df = pd.read_csv(source_path)
    np.random.seed(42)
    raw_df['vehicle_age'] = np.clip((raw_df['age'] * 0.15 + np.random.normal(4, 3, len(raw_df))).round(), 0, 12).astype(int)
    raw_df['annual_mileage'] = np.clip(
        (12000 + (raw_df['vehicle_age'] * 300) - (raw_df['age'] * 15) + np.random.normal(0, 1700, len(raw_df))).round(),
        5000,
        25000,
    ).astype(int)
    raw_df['accident_history'] = np.clip(
        np.random.poisson(0.5, len(raw_df)) + (raw_df['smoker'] == 'yes').astype(int),
        0,
        4,
    ).astype(int)
    raw_df['credit_score'] = np.clip(
        (720 - (raw_df['bmi'] - 26) * 3 - raw_df['accident_history'] * 12 + np.random.normal(0, 35, len(raw_df))).round(),
        300,
        850,
    ).astype(int)
    raw_df['driving_experience'] = np.clip((raw_df['age'] - 16), 0, 50).astype(int)
    raw_df['premium'] = (
        raw_df['charges'] * 0.9
        + raw_df['vehicle_age'] * 85
        + raw_df['accident_history'] * 1100
        - raw_df['credit_score'] * 3
        + np.random.normal(0, 900, len(raw_df))
    ).round(2)
    raw_df['premium'] = raw_df['premium'].clip(lower=1000)
    risk_score = (
        (raw_df['charges'] > raw_df['charges'].quantile(0.65)).astype(int)
        + (raw_df['accident_history'] >= 1).astype(int)
        + (raw_df['smoker'] == 'yes').astype(int)
    )
    raw_df['claim_risk'] = (risk_score >= 2).astype(int)
    cols = [
        'age', 'sex', 'bmi', 'children', 'smoker', 'region',
        'vehicle_age', 'annual_mileage', 'accident_history',
        'credit_score', 'driving_experience', 'claim_risk', 'premium'
    ]
    raw_df[cols].to_csv(csv_path, index=False)
    print('Created new dataset at', csv_path)
else:
    print('Dataset already exists at', csv_path)


Created new dataset at ..\data\insurance_data.csv


In [5]:
# Run the main project pipeline
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve()))

from main import main

main()

Loading or generating dataset...
Dataset loaded: 1338 rows
Preparing features...
Training classification models...


c:\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Training regression models...
Evaluating classification models...
Evaluating regression models...
Creating charts...
Results saved in outputs/
Classification and regression reports are ready.


({'logistic_regression': {'metrics': {'accuracy': 0.9477611940298507,
    'precision': 0.9655172413793104,
    'recall': 0.8235294117647058,
    'f1_score': 0.8888888888888888,
    'roc_auc': np.float64(0.9524999999999999)},
   'report': '              precision    recall  f1-score   support\n\n           0       0.94      0.99      0.97       200\n           1       0.97      0.82      0.89        68\n\n    accuracy                           0.95       268\n   macro avg       0.95      0.91      0.93       268\nweighted avg       0.95      0.95      0.95       268\n'},
  'random_forest': {'metrics': {'accuracy': 0.9664179104477612,
    'precision': 0.9682539682539683,
    'recall': 0.8970588235294118,
    'f1_score': 0.9312977099236641,
    'roc_auc': np.float64(0.9764338235294118)},
   'report': '              precision    recall  f1-score   support\n\n           0       0.97      0.99      0.98       200\n           1       0.97      0.90      0.93        68\n\n    accuracy         